# Phase picking using PhaseNet (INSTANCE pre-trained model)

This notebook applies the PhaseNet model pre-trained on the INSTANCE dataset to detect P-wave and S-wave arrivals in the accelerometric signals.

## Imports

In [ ]:
import seisbench
import seisbench.models as sbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add src to path
sys.path.append('../src')
from src.phasenet_utils import (
    get_station_from_filename, 
    get_component_from_filename, 
    create_obspy_stream_from_dataframe
)

## Data loading

In [ ]:
df_signals = pd.read_parquet('../data/processed/acc_preprocessed_scaling.parquet')
df_meta = pd.read_parquet('../data/processed/metadata_clean.parquet')

print(f"Loaded {len(df_signals)} signal samples")
print(f"Loaded {len(df_meta)} metadata entries")

## Load PhaseNet model pre-trained on INSTANCE

In [ ]:
model = sbm.PhaseNet.from_pretrained("instance")
print("PhaseNet model loaded successfully")

## Phase picking with resampling to 100 Hz

In [ ]:
results = []
sampling_rate_original = 200  # Hz
sampling_rate_target = 100    # Hz for PhaseNet

for station_code in df_signals['file'].apply(get_station_from_filename).unique():
    mask = df_signals['file'].apply(lambda f: get_station_from_filename(f) == station_code)
    df_station = df_signals[mask].copy()
    
    # Create ObsPy Stream
    stream, comp_names = create_obspy_stream_from_dataframe(
        df_station, station_code, sampling_rate_original
    )
    
    if stream is None:
        print(f"Skipping {station_code}: incomplete components")
        continue
    
    original_starttime = stream[0].stats.starttime
    original_npts = stream[0].stats.npts
    original_duration = original_npts / sampling_rate_original
    
    # PhaseNet expects ~30s windows at 100 Hz
    expected_samples_100hz = 3001
    expected_duration = expected_samples_100hz / sampling_rate_target
    
    if original_duration < expected_duration:
        print(f"Skipping {station_code}: signal too short ({original_duration:.1f}s < {expected_duration:.1f}s)")
        continue
    
    # Resample to 100 Hz
    stream_resampled = stream.copy()
    stream_resampled.resample(sampling_rate_target)
    
    resampled_duration = stream_resampled[0].stats.npts / sampling_rate_target
    duration_diff = abs(original_duration - resampled_duration)
    if duration_diff > 0.5:
        print(f"Warning {station_code}: resampling changed duration by {duration_diff:.2f}s")
    
    # Apply PhaseNet
    annotations = model.annotate(stream_resampled)
    
    # Extract P and S probability traces
    p_traces = annotations.select(channel="PhaseNet_P")
    s_traces = annotations.select(channel="PhaseNet_S")
    
    if len(p_traces) == 0 or len(s_traces) == 0:
        print(f"Skipping {station_code}: no annotations produced")
        continue
    
    p_trace = p_traces[0]
    s_trace = s_traces[0]
    
    # Get probability arrays
    p_prob = p_trace.data
    s_prob = s_trace.data
    
    # Find onset as maximum probability
    p_idx_resampled = p_prob.argmax()
    s_idx_resampled = s_prob.argmax()
    
    # Time relative to annotation trace start
    t_p_annotation = p_idx_resampled / sampling_rate_target
    t_s_annotation = s_idx_resampled / sampling_rate_target
    
    time_offset = float(p_trace.stats.starttime - original_starttime)
    
    # Time relative to ORIGINAL stream start
    t_p_original = t_p_annotation + time_offset
    t_s_original = t_s_annotation + time_offset
    
    p_idx_original = int(round(t_p_original * sampling_rate_original))
    s_idx_original = int(round(t_s_original * sampling_rate_original))
    
    if p_idx_original < 0 or p_idx_original >= original_npts:
        print(f"Warning {station_code}: P pick outside signal bounds ({p_idx_original}/{original_npts})")
    if s_idx_original < 0 or s_idx_original >= original_npts:
        print(f"Warning {station_code}: S pick outside signal bounds ({s_idx_original}/{original_npts})")
    
    # Save results
    results.append({
        'station_code': station_code,
        'components': ', '.join(comp_names),
        't_p_samples': p_idx_original,
        't_s_samples': s_idx_original,
        't_p_seconds': float(t_p_original),
        't_s_seconds': float(t_s_original),
        'p_probability_max': float(p_prob[p_idx_resampled]),
        's_probability_max': float(s_prob[s_idx_resampled]),
        'time_offset_seconds': float(time_offset),
        'original_duration_s': float(original_duration),
        'annotation_npts': len(p_prob)
    })
    
    print(f"{station_code}: P={t_p_original:.2f}s, S={t_s_original:.2f}s, offset={time_offset:.2f}s")

# %% Convert to DataFrame
df_picks = pd.DataFrame(results)

# %% Quality check: S should come after P
invalid_picks = df_picks[df_picks['t_s_seconds'] <= df_picks['t_p_seconds']]
if len(invalid_picks) > 0:
    print(f"\nWARNING: {len(invalid_picks)} stations have S before or at P:")
    print(invalid_picks[['station_code', 't_p_seconds', 't_s_seconds']])
else:
    print("\nAll picks are physically plausible (S > P)")

# %% Save results
output_path = Path('../data/processed/phasenet_picks_instance.csv')
df_picks.to_csv(output_path, index=False)
print(f"\nProcessed {len(results)} stations")
print(f"Results saved to {output_path}")

# %% Display summary statistics
print("\n=== SUMMARY ===")
print(f"Total stations processed: {len(df_picks)}")
print(f"Mean P probability: {df_picks['p_probability_max'].mean():.3f}")
print(f"Mean S probability: {df_picks['s_probability_max'].mean():.3f}")
print(f"Mean S-P time: {(df_picks['t_s_seconds'] - df_picks['t_p_seconds']).mean():.2f}s")